# Compass SP-2 — clinical placement CSP
**C2-primary | synthetic teaching data | 25 September 2026**  
Owner: **Anoop G.** · Reviewer: **Asghar K.** · Squad: **LI-2**

Run all cells from the directory containing this notebook and `csp_toy_instance.json`. Requires Python 3 and SciPy (`scipy.optimize.milp`). The notebook writes the template-shaped primary JSON and the infeasibility diagnosis JSON. No real student data is used.


## 1. Formulation sheet
Let $I$ be students, $B=\{1,2\}$ blocks, $J$ sites, and $R_i$ the two required specialties of student $i$. A candidate $p=(j_1,r_1,j_2,r_2)$ is a two-block plan. Let $P_i$ contain plans whose two specialties are exactly $R_i$, each site offers the assigned specialty, sites differ, prerequisites and minimum hours hold, and each travel time is at most $L=45$. Binary $y_{ip}$ selects a plan. $a_{ipbj}=1$ when plan $p$ puts student $i$ at site $j$ in block $b$, and $t_{ip}$ is its two-block travel.

| Item | Formal rule | Plain English |
|---|---|---|
| Variables and domains | $y_{ip}\in\{0,1\}$ for $i\in I,p\in P_i$; equivalently $(x_{i1},x_{i2})\in J^2$ with specialty labels | Choose exactly one eligible two-block placement for each student. |
| Specialty coverage | $\{r_{i1},r_{i2}\}=R_i$ and $r_{ib}\in S_{x_{ib}}$ | Cover both required specialties, each at a site offering it. |
| No repeat site | $x_{i1}\ne x_{i2}$ | A student visits two different sites. |
| Capacity | $\sum_i\sum_{p\in P_i}a_{ipbj}y_{ip}\le c_j$ for every $b,j$ | Never exceed a site's block capacity. |
| Prerequisite | $q_j=\varnothing$ or $q_j\in Q_i$ whenever $x_{ib}=j$ | Students must have completed a site's prerequisite. |
| Licensure hours | $h_i\ge m_j$ whenever $x_{ib}=j$ | Students must meet a site's minimum hours. |
| Travel limit | $d_{ij}\le L$ whenever $x_{ib}=j$ | No placement exceeds 45 minutes of travel. |
| Exactly one plan | $\sum_{p\in P_i}y_{ip}=1$ for every $i$ | Every student gets both blocks. |
| Fairness (soft) | Minimize $M=\max_i\sum_{p\in P_i}t_{ip}y_{ip}$ | Minimize the worst two-block travel burden on any student. |
| Tie-break | Among schedules with minimum $M$, minimize $T=\sum_i\sum_p t_{ip}y_{ip}$ | For equal worst burdens, prefer less travel overall. |

Meridian should endorse or change the minimax priority explicitly: it protects the most burdened student but can increase cohort travel. Both scores are reported so that tradeoff is visible. This is a proposed design choice, not a claimed Meridian policy.


In [1]:
import json
from pathlib import Path
import numpy as np
from scipy.optimize import milp, Bounds, LinearConstraint
from scipy.sparse import lil_matrix

SQUAD = "LI-2"
DATA = Path("csp_toy_instance.json")
assets = json.loads(DATA.read_text())
primary, infeasible = (next(x for x in assets["instances"] if x["id"] == key)
                       for key in ("C2-primary", "C2-infeasible"))
LIMIT = assets["travel_limit_minutes"]
print("Loaded", primary["id"], "and", infeasible["id"], "with travel limit", LIMIT)


Loaded C2-primary and C2-infeasible with travel limit 45


In [2]:
def eligible_plans(instance):
    sites = {s["id"]: s for s in instance["sites"]}
    plans = {}
    for student in instance["students"]:
        sid = student["id"]
        options = []
        for first, second in (student["required_specialties"], student["required_specialties"][::-1]):
            for j1 in sites:
                for j2 in sites:
                    if j1 == j2:
                        continue
                    def eligible(j, specialty):
                        site = sites[j]
                        return (specialty in site["specialties"]
                                and (site["requires_prerequisite"] is None or
                                     site["requires_prerequisite"] in student["completed_prerequisites"])
                                and student["licensure_hours"] >= site["min_licensure_hours"]
                                and instance["travel_minutes"][sid][j] <= LIMIT)
                    if eligible(j1, first) and eligible(j2, second):
                        options.append(((j1, first), (j2, second)))
        plans[sid] = list(dict.fromkeys(options))
    return plans

def solve(instance):
    plans = eligible_plans(instance)
    students = [s["id"] for s in instance["students"]]
    index = [(i, p) for i in students for p in plans[i]]
    n = len(index)
    # Final variable M is the maximum student's combined travel.
    rows, lower, upper = [], [], []
    for i in students:
        rows.append({k: 1 for k, (sid, _) in enumerate(index) if sid == i})
        lower.append(1); upper.append(1)
    for block in (0, 1):
        for site in instance["sites"]:
            rows.append({k: 1 for k, (_, p) in enumerate(index) if p[block][0] == site["id"]})
            lower.append(-np.inf); upper.append(site["capacity_per_block"])
    for i in students:
        row = {k: sum(instance["travel_minutes"][i][part[0]] for part in p)
               for k, (sid, p) in enumerate(index) if sid == i}
        row[n] = -1
        rows.append(row); lower.append(-np.inf); upper.append(0)
    A = lil_matrix((len(rows), n + 1))
    for r, row in enumerate(rows):
        for k, value in row.items(): A[r, k] = value
    bounds = Bounds(np.zeros(n+1), np.r_[np.ones(n), np.inf])
    integer = np.r_[np.ones(n), 0]
    constraints = LinearConstraint(A.tocsr(), lower, upper)
    objective = np.r_[np.zeros(n), 1.]
    stage1 = milp(objective, integrality=integer, bounds=bounds, constraints=constraints,
                  options={"time_limit": 60, "mip_rel_gap": 0})
    if stage1.status != 0:
        return stage1, None, plans
    optimum = int(round(stage1.x[n]))
    # Exact lexicographic second stage; M cannot exceed the proven minimum.
    objective2 = np.r_[[sum(instance["travel_minutes"][sid][part[0]] for part in p)
                         for sid, p in index], 0.]
    bounds2 = Bounds(np.zeros(n+1), np.r_[np.ones(n), optimum])
    stage2 = milp(objective2, integrality=integer, bounds=bounds2, constraints=constraints,
                  options={"time_limit": 60, "mip_rel_gap": 0})
    assert stage2.status == 0, f"Tie-break solve incomplete: {stage2.message}"
    assignment = {}
    for k, (sid, p) in enumerate(index):
        if stage2.x[k] > .5:
            assignment[sid] = {f"block_{b+1}": {"site": j, "specialty": specialty}
                               for b, (j, specialty) in enumerate(p)}
    assert len(assignment) == len(students)
    return stage2, assignment, plans

result, assignment, plans = solve(primary)
assert result.status == 0, result.message
print("Primary solver status:", result.message)
print(json.dumps(assignment, indent=2))


Primary solver status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
{
  "s01": {
    "block_1": {
      "site": "riverside",
      "specialty": "family_medicine"
    },
    "block_2": {
      "site": "harborview",
      "specialty": "pediatrics"
    }
  },
  "s02": {
    "block_1": {
      "site": "riverside",
      "specialty": "family_medicine"
    },
    "block_2": {
      "site": "harborview",
      "specialty": "pediatrics"
    }
  },
  "s03": {
    "block_1": {
      "site": "st_augustine",
      "specialty": "family_medicine"
    },
    "block_2": {
      "site": "riverside",
      "specialty": "pediatrics"
    }
  },
  "s04": {
    "block_1": {
      "site": "northgate",
      "specialty": "rehabilitation"
    },
    "block_2": {
      "site": "st_augustine",
      "specialty": "geriatrics"
    }
  },
  "s05": {
    "block_1": {
      "site": "st_augustine",
      "specialty": "geriatrics"
    },
    "block_2": {
      "site": "northgate",
      "specialty": 

In [3]:
def verify(instance, assignment):
    sites = {x["id"]: x for x in instance["sites"]}
    students = {x["id"]: x for x in instance["students"]}
    checks = {name: [] for name in ("student/block completeness", "specialty coverage",
              "no repeated site", "site capacity", "prerequisite completion",
              "licensure-hour minimum", "travel limit")}
    checks["student/block completeness"].append(set(assignment) == set(students))
    counts = {(b, site): 0 for b in instance["blocks"] for site in sites}
    burdens = {}
    for sid, student in students.items():
        blocks = assignment.get(sid, {})
        complete = set(blocks) == set(instance["blocks"])
        checks["student/block completeness"].append(complete)
        if not complete: continue
        chosen = [blocks[b] for b in instance["blocks"]]
        known = all(p.get("site") in sites for p in chosen)
        checks["student/block completeness"].append(known)
        if not known: continue
        checks["specialty coverage"].append(
            sorted(p["specialty"] for p in chosen) == sorted(student["required_specialties"])
            and all(p["specialty"] in sites[p["site"]]["specialties"] for p in chosen))
        checks["no repeated site"].append(chosen[0]["site"] != chosen[1]["site"])
        burdens[sid] = 0
        for b, p in zip(instance["blocks"], chosen):
            j = p["site"]; site = sites[j]; counts[b, j] += 1
            checks["prerequisite completion"].append(site["requires_prerequisite"] is None
                or site["requires_prerequisite"] in student["completed_prerequisites"])
            checks["licensure-hour minimum"].append(
                student["licensure_hours"] >= site["min_licensure_hours"])
            travel = instance["travel_minutes"][sid][j]
            checks["travel limit"].append(travel <= LIMIT)
            burdens[sid] += travel
    for (b, j), count in counts.items():
        checks["site capacity"].append(count <= sites[j]["capacity_per_block"])
    status = {name: all(values) and bool(values) for name, values in checks.items()}
    assert all(status.values()), f"Hard constraint failure: {status}"
    return status, burdens, counts

checks, burdens, counts = verify(primary, assignment)
for name, passed in checks.items(): print(f"{name}: {'PASS' if passed else 'FAIL'}")
total = sum(burdens.values()); fairness = max(burdens.values())
print("Total travel minutes:", total)
print("Fairness score (maximum student two-block minutes):", fairness)
print("Student burdens:", burdens)
print("Block/site occupancy:", counts)


student/block completeness: PASS
specialty coverage: PASS
no repeated site: PASS
site capacity: PASS
prerequisite completion: PASS
licensure-hour minimum: PASS
travel limit: PASS
Total travel minutes: 575
Fairness score (maximum student two-block minutes): 74
Student burdens: {'s01': 54, 's02': 33, 's03': 56, 's04': 74, 's05': 59, 's06': 35, 's07': 34, 's08': 53, 's09': 44, 's10': 37, 's11': 34, 's12': 62}
Block/site occupancy: {('block_1', 'riverside'): 4, ('block_1', 'northgate'): 3, ('block_1', 'harborview'): 2, ('block_1', 'st_augustine'): 3, ('block_2', 'riverside'): 4, ('block_2', 'northgate'): 3, ('block_2', 'harborview'): 2, ('block_2', 'st_augustine'): 3}


In [4]:
submission = {
    "squad": SQUAD, "instance": "C2-primary",
    "fairness_objective": {
        "name": "minimax_student_travel",
        "description": "Minimize the largest two-block travel borne by any single student.",
        "reported_value": fairness,
    },
    "reported_total_travel_minutes": total,
    "assignment": assignment,
}
output = Path(f"{SQUAD}.json")
output.write_text(json.dumps(submission, indent=2) + "\n")
assert json.loads(output.read_text())["fairness_objective"]["reported_value"] == fairness
print("Wrote", output)


Wrote MA-1.json


In [5]:
bad_result, bad_assignment, bad_plans = solve(infeasible)
print("Infeasible instance solver status:", bad_result.status, bad_result.message)
assert bad_result.status == 2 and bad_assignment is None, "Expected proven infeasibility"
# Explain a necessary bottleneck, separately from the solver's status.
# s03 and s11 both require family medicine and can only use Riverside for pediatrics/geriatrics
# respectively; inspect their feasible plans and site occupancy after the capacity change.
for sid in ("s03", "s11"):
    print(sid, "eligible plans:", bad_plans[sid])
print("St. Augustine capacity per block:", next(s["capacity_per_block"] for s in infeasible["sites"] if s["id"] == "st_augustine"))


Infeasible instance solver status: 2 The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)
s03 eligible plans: [(('st_augustine', 'family_medicine'), ('riverside', 'pediatrics')), (('riverside', 'pediatrics'), ('st_augustine', 'family_medicine'))]
s11 eligible plans: [(('riverside', 'family_medicine'), ('st_augustine', 'geriatrics')), (('st_augustine', 'geriatrics'), ('riverside', 'family_medicine'))]
St. Augustine capacity per block: 2


In [6]:
diagnosis = {
    "squad": SQUAD, "instance": "C2-infeasible", "feasible": False,
    "diagnosis": "Full hard-constraint model is infeasible after St. Augustine capacity drops from 3 to 2 per block; eligibility, specialty coverage, distinct sites, and block capacity cannot all be satisfied.",
    "proposed_relaxations": [
        {"change": "Restore one St. Augustine preceptor slot in each block after site approval.",
         "cost": "Site funds and supervises one additional placement per block; amount to be confirmed."},
        {"change": "Add an approved substitute site with needed specialties and adequate capacity, then resolve.",
         "cost": "Registrar coordinates placements; affected students may bear additional travel, to be measured."}
    ]
}
Path(f"{SQUAD}-infeasible.json").write_text(json.dumps(diagnosis, indent=2) + "\n")
print("Wrote", f"{SQUAD}-infeasible.json")


Wrote MA-1-infeasible.json
